# KVO_02 — Four-site audit + external validation of the MOS_1_2 U-Net

This notebook establishes the multisite UAV benchmark **before retraining**. It audits MOS_1_2, Yoko, Baego and Yangambi; then applies the frozen KVO_01 MOS_1_2 U-Net unchanged to the three independent sites.

Primary questions: (1) are raster/annotation geometry and labels compatible across sites? (2) how well does the MOS_1_2 model transfer to each new site at labelled-pixel and crown level? (3) what full-resolution probability maps does it produce? **No multisite fitting occurs here.** `Class_ID == 1` is Musanga; all other classified crowns are Other; unlabelled pixels are excluded from quantitative evaluation.

In [ ]:
!pip -q install geopandas rasterio pyogrio segmentation-models-pytorch scikit-learn matplotlib tqdm
from pathlib import Path
import json, shutil, time, warnings
import numpy as np, pandas as pd, geopandas as gpd, matplotlib.pyplot as plt
import rasterio
from rasterio.features import rasterize
from rasterio.windows import Window
from sklearn.metrics import roc_auc_score, average_precision_score, precision_recall_curve, confusion_matrix
from tqdm.auto import tqdm
import torch
import segmentation_models_pytorch as smp
from google.colab import drive
warnings.filterwarnings('ignore')
SEED=42
np.random.seed(SEED); torch.manual_seed(SEED)
DEVICE='cuda' if torch.cuda.is_available() else 'cpu'
torch.backends.cudnn.benchmark=True
print('DEVICE:',DEVICE)
if DEVICE=='cuda': print(torch.cuda.get_device_name(0))

In [ ]:
DRIVE_ROOT=Path('/content/drive'); MYDRIVE=DRIVE_ROOT/'MyDrive'
def drive_is_healthy():
    try: return MYDRIVE.exists() and any(MYDRIVE.iterdir())
    except OSError: return False
if not drive_is_healthy(): drive.mount(str(DRIVE_ROOT),force_remount=True)
ROOT=MYDRIVE/'Colab Notebooks'/'Musanga'; GEO=ROOT/'data'/'Geo_Ref'; ANN=GEO/'Annotations'
OUT=ROOT/'KVO_work'/'KVO_02_external_validation'; OUT.mkdir(parents=True,exist_ok=True)
SITES={
 'MOS_1_2': {'raster':GEO/'MOS_1_2_kvo.tif','annotations':ANN/'Tree_Annotations_MOS_1_2_kvo.shp'},
 'Yoko': {'raster':GEO/'Yoko_kvo_georef.tif','annotations':ANN/'Tree_Annotations_Yoko_kvo_georef.shp'},
 'Baego': {'raster':GEO/'Baego_georef.tif','annotations':ANN/'Tree_Annotation_Baego.shp'},
 'Yangambi': {'raster':GEO/'Yangambi_georef.tif','annotations':ANN/'Tree_Annotation_Yangambi_georef.shp'},
}
MODEL_PATH=ROOT/'KVO_work'/'MOS_1_2_pixel_unet'/'model'/'MOS_1_2_unet_resnet18_best.pt'
for site,d in SITES.items():
    assert d['raster'].exists(),d['raster']; assert d['annotations'].exists(),d['annotations']
assert MODEL_PATH.exists(),MODEL_PATH
print('All four raster/annotation pairs + KVO_01 checkpoint found.')

## 1 — Four-site audit

Checks raster size, bands, dtype, GSD, CRS, annotation CRS, `Class_ID`, class counts and annotation/raster overlap. This cell does not alter any data.

In [ ]:
audit=[]; ANNS={}
for site,d in SITES.items():
    with rasterio.open(d['raster']) as src:
        crs=src.crs; bounds=src.bounds
        info={'site':site,'raster_gb':d['raster'].stat().st_size/1e9,'width':src.width,'height':src.height,'bands':src.count,'dtype':src.dtypes[0],
              'gsd_x_m':abs(src.transform.a),'gsd_y_m':abs(src.transform.e),'crs':str(crs)}
    g=gpd.read_file(d['annotations'])
    assert 'Class_ID' in g.columns, f'{site}: Class_ID missing; fields={list(g.columns)}'
    source_crs=str(g.crs)
    if g.crs!=crs: g=g.to_crs(crs)
    g=g[g.geometry.notna() & ~g.geometry.is_empty].copy()
    g['class_id_num']=pd.to_numeric(g['Class_ID'],errors='coerce'); g=g[g.class_id_num.notna()].copy(); g['class_id_num']=g.class_id_num.astype(int)
    g['target']=(g.class_id_num==1).astype(np.uint8); g['crown_id']=np.arange(len(g),dtype=np.int64)
    rb=gpd.GeoSeries([__import__('shapely').geometry.box(*bounds)],crs=crs)
    overlap=g.geometry.intersects(rb.iloc[0])
    info.update({'annotation_crs_original':source_crs,'n_crowns':len(g),'n_musanga':int(g.target.sum()),'n_other':int((1-g.target).sum()),
                 'crowns_intersect_raster':int(overlap.sum()),'overlap_pct':float(100*overlap.mean()),'class_ids':','.join(map(str,sorted(g.class_id_num.unique())))})
    ANNS[site]=g; audit.append(info)
audit_df=pd.DataFrame(audit); audit_df.to_csv(OUT/'site_audit.csv',index=False)
display(audit_df)
print('\nPer-site Class_ID counts:')
for site,g in ANNS.items(): print('\n',site,'\n',g.class_id_num.value_counts().sort_index().to_string())

## 2 — Load the frozen MOS_1_2 model

Exactly the KVO_01 U-Net/ResNet18 architecture and checkpoint. No weights are updated in this notebook.

In [ ]:
MEAN=np.array([0.485,0.456,0.406],np.float32)[:,None,None]; STD=np.array([0.229,0.224,0.225],np.float32)[:,None,None]
model=smp.Unet(encoder_name='resnet18',encoder_weights=None,in_channels=3,classes=1).to(DEVICE)
model.load_state_dict(torch.load(MODEL_PATH,map_location=DEVICE)); model.eval()
print('Frozen checkpoint loaded:',MODEL_PATH)

## 3 — External-site labelled-pixel evaluation

Predicts only windows around annotated crowns, so this is much faster than creating full maps. Pixels inside digitized crowns are evaluated as Musanga (`Class_ID=1`) versus Other. Crown overlap is resolved by rasterization order; inspect the audit if overlapping polygons are common. MOS_1_2 is omitted here because its proper spatially held-out KVO_01 result already exists.

In [ ]:
PATCH=512; HALO=64; IGNORE=255
def predict_window(src,win):
    arr=src.read([1,2,3],window=win,boundless=True,fill_value=0).astype(np.float32)
    # KVO_01 assumes 8-bit RGB. Stop rather than silently truncate a different source encoding.
    if np.nanmax(arr)>255: raise ValueError(f'{src.name}: RGB values exceed 255; normalize explicitly before applying KVO_01 model.')
    arr/=255.; ph=(32-arr.shape[1]%32)%32; pw=(32-arr.shape[2]%32)%32
    arr=np.pad(arr,((0,0),(0,ph),(0,pw))); x=torch.from_numpy((arr-MEAN)/STD).unsqueeze(0).to(DEVICE)
    with torch.inference_mode(),torch.amp.autocast('cuda',enabled=(DEVICE=='cuda')): p=torch.sigmoid(model(x).squeeze()).float().cpu().numpy()
    return p[:win.height,:win.width]

def evaluate_site(site):
    rp=SITES[site]['raster']; g=ANNS[site]; yy=[]; pp=[]; crown_rows=[]
    with rasterio.open(rp) as src:
        for row in tqdm(g.itertuples(),total=len(g),desc=f'{site} crowns'):
            geom=row.geometry
            w=rasterio.windows.from_bounds(*geom.bounds,transform=src.transform).round_offsets().round_lengths()
            w=w.intersection(Window(0,0,src.width,src.height))
            if w.width<1 or w.height<1: continue
            # Predict crown bounding window plus context.
            ew=Window(w.col_off-HALO,w.row_off-HALO,w.width+2*HALO,w.height+2*HALO)
            pfull=predict_window(src,ew); p=pfull[HALO:HALO+int(w.height),HALO:HALO+int(w.width)]
            tr=src.window_transform(w); inside=rasterize([(geom,1)],out_shape=(int(w.height),int(w.width)),transform=tr,fill=0,dtype='uint8').astype(bool)
            v=p[inside]
            if not len(v): continue
            yy.append(np.full(len(v),row.target,np.uint8)); pp.append(v.astype(np.float32))
            crown_rows.append({'site':site,'crown_id':row.crown_id,'class_id':row.class_id_num,'truth':int(row.target),'n_pixels':len(v),
                               'prob_mean':float(v.mean()),'prob_median':float(np.median(v)),'prob_p90':float(np.quantile(v,.9))})
    y=np.concatenate(yy); p=np.concatenate(pp); c=pd.DataFrame(crown_rows)
    met={'site':site,'n_labelled_pixels':int(len(y)),'n_crowns':int(len(c)),'n_musanga':int(c.truth.sum()),'n_other':int((1-c.truth).sum()),
         'pixel_roc_auc':float(roc_auc_score(y,p)),'pixel_average_precision':float(average_precision_score(y,p)),
         'crown_roc_auc_mean_probability':float(roc_auc_score(c.truth,c.prob_mean)),'crown_average_precision_mean_probability':float(average_precision_score(c.truth,c.prob_mean))}
    return met,c

# Resume-safe external evaluation.
# IMPORTANT: changing this GitHub notebook does not affect a copy already running in Colab.
FORCE_EXTERNAL_EVAL=False
METRICS_CSV=OUT/'external_site_metrics.csv'
CROWNS_CSV=OUT/'external_crown_probabilities.csv'

if METRICS_CSV.exists() and CROWNS_CSV.exists() and not FORCE_EXTERNAL_EVAL:
    print('Existing external-evaluation results found — SKIPPING crown inference.')
    print('Loading:',METRICS_CSV)
    print('Loading:',CROWNS_CSV)
    external_metrics=pd.read_csv(METRICS_CSV)
    external_crowns=pd.read_csv(CROWNS_CSV)
else:
    metrics=[]; crowns=[]
    for site in ['Yoko','Baego','Yangambi']:
        m,c=evaluate_site(site)
        metrics.append(m); crowns.append(c)
        print(json.dumps(m,indent=2))
    external_metrics=pd.DataFrame(metrics)
    external_crowns=pd.concat(crowns,ignore_index=True)
    # Atomic-ish writes: write temporary files first, then replace.
    metrics_tmp=OUT/'external_site_metrics.tmp.csv'
    crowns_tmp=OUT/'external_crown_probabilities.tmp.csv'
    external_metrics.to_csv(metrics_tmp,index=False)
    external_crowns.to_csv(crowns_tmp,index=False)
    metrics_tmp.replace(METRICS_CSV)
    crowns_tmp.replace(CROWNS_CSV)
    print('Saved reusable external-evaluation cache.')

display(external_metrics)


## 4 — Cross-site crown probability distributions

A compact visual check of transfer: Musanga should remain concentrated toward 1 and Other toward 0 on each independent site.

In [ ]:
fig,ax=plt.subplots(figsize=(9,5)); bins=np.linspace(0,1,31)
for site in ['Yoko','Baego','Yangambi']:
    v=external_crowns[(external_crowns.site==site)&(external_crowns.truth==1)].prob_mean
    ax.hist(v,bins=bins,histtype='step',density=True,label=f'{site} Musanga (n={len(v)})')
ax.set(xlim=(0,1),xlabel='Mean Musanga probability within crown',ylabel='Density',title='Frozen MOS_1_2 model — Musanga crowns on independent sites'); ax.legend(); fig.tight_layout()
fig.savefig(OUT/'external_musanga_crown_probability_histogram.png',dpi=180); plt.show()

fig,ax=plt.subplots(figsize=(9,5))
for site in ['Yoko','Baego','Yangambi']:
    v=external_crowns[(external_crowns.site==site)&(external_crowns.truth==0)].prob_mean
    ax.hist(v,bins=bins,histtype='step',density=True,label=f'{site} Other (n={len(v)})')
ax.set(xlim=(0,1),xlabel='Mean Musanga probability within crown',ylabel='Density',title='Frozen MOS_1_2 model — Other crowns on independent sites'); ax.legend(); fig.tight_layout()
fig.savefig(OUT/'external_other_crown_probability_histogram.png',dpi=180); plt.show()

## 5 — Full-resolution probability maps for visual QA

After the fast external benchmark, create georeferenced probability rasters for the three new sites. Outputs use the KVO_01 convention: uint16 `0–10000 = probability 0–1`, NoData `65535`. Existing complete outputs are reused. Large mosaics are read directly from Drive to avoid requiring enough Colab SSD for all four sites.

In [ ]:
PROB=OUT/'probability'; PROB.mkdir(exist_ok=True); CORE=512; HALO=64
def full_inference(site,force=False):
    rp=SITES[site]['raster']; out=PROB/f'{site}_MOS_1_2model_musanga_probability_uint16.tif'
    if out.exists() and out.stat().st_size>0 and not force:
        print(site,'existing output — skipping:',out); return out
    with rasterio.open(rp) as src:
        H,W=src.height,src.width
        if src.count<3: raise ValueError(f'{site}: requires >=3 RGB bands')
        if not all(np.dtype(x)==np.dtype('uint8') for x in src.dtypes[:3]): raise ValueError(f'{site}: KVO_01 expects uint8 RGB; got {src.dtypes[:3]}')
        profile=src.profile.copy(); profile.update(count=1,dtype='uint16',nodata=65535,compress='deflate',predictor=2,tiled=True,BIGTIFF='YES')
        with rasterio.open(out,'w',**profile) as dst:
            jobs=[(r,c,min(CORE,H-r),min(CORE,W-c)) for r in range(0,H,CORE) for c in range(0,W,CORE)]
            for r,c,hh,ww in tqdm(jobs,desc=f'{site} full mosaic'):
                win=Window(c-HALO,r-HALO,ww+2*HALO,hh+2*HALO)
                p=predict_window(src,win); core=p[HALO:HALO+hh,HALO:HALO+ww]
                dst.write(np.clip(np.round(core*10000),0,10000).astype(np.uint16),1,window=Window(c,r,ww,hh))
    print('Saved:',out); return out

FULL_MAPS={site:str(full_inference(site)) for site in ['Yoko','Baego','Yangambi']}
FULL_MAPS

## 6 — Summary and next decision

This notebook deliberately preserves the three new mosaics as genuinely external tests of the MOS_1_2 model. Once these transfer results and maps are inspected, the next notebook can construct a **multisite training experiment** using all four sites, with site-aware validation (ideally leave-one-site-out) rather than mixing crowns randomly across mosaics.

In [ ]:
summary={'purpose':'Frozen MOS_1_2 U-Net external validation before multisite retraining','label_rule':'Class_ID == 1 is Musanga; other classified crowns are Other','checkpoint':str(MODEL_PATH),
         'sites':{k:{'raster':str(v['raster']),'annotations':str(v['annotations'])} for k,v in SITES.items()},
         'external_metrics':external_metrics.to_dict(orient='records'),'probability_maps':FULL_MAPS}
(OUT/'KVO_02_summary.json').write_text(json.dumps(summary,indent=2))
print('Outputs:',OUT); print('\nDONE — independent-site benchmark + probability maps; no retraining performed.')